# Unified CSBS Scoring Validation Notebook

**Purpose**: Validate both CSBS instruments across two separate REDCap sandboxes:
1. **CSBS Caregiver + EDI-YC**: PID 6205 (NANO - CSBS Caregiver / EDI Scoring Sandbox)
2. **CSBS Baby Siblings (BS)**: PID 6207 (NANO Lab Assessments & Double Data Entry Sandbox)

**Scope**:
- Auto-calculated score formulas match approved logic
- All raw/composite/total score fields properly labeled with `AUTO-CALCULATED:` prefix
- Field visibility and highlighting in Online Designer
- Consistency across both instruments (labeling, formula hierarchy, rounding)
- Manual norm fields present and scoped to correct visits

**Tokens Required**:
- `REDCAP_TOKEN_6205`: CSBS Caregiver sandbox (set via environment)
- `REDCAP_TOKEN_6207`: CSBS BS sandbox (defaults to prod token)

**Note**: This is validation-only and does not derive or implement new logic.

In [1]:
import os
import requests
import math
from decimal import Decimal, ROUND_HALF_UP

API_URL = os.getenv('REDCAP_API_URL', 'https://redcap.research.sc.edu/api/')
TOKEN_6205 = os.getenv('REDCAP_TOKEN_6205', '').strip()  # CSBS Caregiver
TOKEN_6207 = os.getenv('REDCAP_TOKEN_6207', '7B81ED83E91654B3573A2D7FE4B223D2').strip()  # CSBS BS

print('\\n' + '='*70)
print('UNIFIED CSBS VALIDATION - INITIALIZATION')
print('='*70)
print(f'API Endpoint: {API_URL}')
print(f'Token 6205 (Caregiver): {"Set" if TOKEN_6205 else "NOT SET - will skip PID 6205"}')
print(f'Token 6207 (BS): {"Set" if TOKEN_6207 else "NOT SET - cannot proceed"}')

if not TOKEN_6207:
    raise RuntimeError('REDCAP_TOKEN_6207 must be set for CSBS BS validation.')

def post(token, content, **params):
    """Make REDCap API call."""
    data = {'token': token, 'content': content, 'format': 'json', 'returnFormat': 'json'}
    data.update(params)
    r = requests.post(API_URL, data=data, timeout=180)
    r.raise_for_status()
    payload = r.json()
    if isinstance(payload, dict) and payload.get('error'):
        raise RuntimeError(payload['error'])
    return payload

# Verify projects
print('\\n--- Verifying Projects ---')
proj_6207 = post(TOKEN_6207, 'project')
proj_6207 = proj_6207[0] if isinstance(proj_6207, list) and proj_6207 else proj_6207
if str(proj_6207.get('project_id')) != '6207':
    raise RuntimeError('TOKEN_6207 does not map to PID 6207')
print(f'✓ PID 6207: {proj_6207.get("project_title")}')

if TOKEN_6205:
    proj_6205 = post(TOKEN_6205, 'project')
    proj_6205 = proj_6205[0] if isinstance(proj_6205, list) and proj_6205 else proj_6205
    if str(proj_6205.get('project_id')) != '6205':
        raise RuntimeError('TOKEN_6205 does not map to PID 6205')
    print(f'✓ PID 6205: {proj_6205.get("project_title")}')
else:
    print(f'⚠ PID 6205: SKIPPED (token not set)')

\n======================================================================
UNIFIED CSBS VALIDATION - INITIALIZATION
API Endpoint: https://redcap.research.sc.edu/api/
Token 6205 (Caregiver): NOT SET - will skip PID 6205
Token 6207 (BS): Set
\n--- Verifying Projects ---


✓ PID 6207: NANO Lab Assessments & Double Data Entry- Sandbox
⚠ PID 6205: SKIPPED (token not set)


In [2]:
# Utility functions
def num(v):
    """Safe numeric conversion."""
    if v in (None, ''):
        return None
    try:
        return float(v)
    except (TypeError, ValueError):
        return None

def half_up(v):
    """Round using ROUND_HALF_UP (REDCap default)."""
    return float(Decimal(str(v)).quantize(Decimal('1'), rounding=ROUND_HALF_UP))

def ceil_sum(row, fields, transforms=None):
    """Sum with ceil (CSBS Caregiver domain scoring)."""
    values = [num(row.get(f)) for f in fields]
    if any(v is None for v in values):
        return None
    transforms = transforms or {}
    adjusted = [transforms.get(f, lambda x: x)(v) for f, v in zip(fields, values)]
    return float(math.ceil(sum(adjusted) - 1e-12))

def bs_expected(row):
    """Calculate expected CSBS BS scores."""
    required = [f'csbsbs_scale{i}' for i in range(1, 16)] + [
        'csbsbs_scale16_1', 'csbsbs_scale16_2', 'csbsbs_scale16_3',
        'csbsbs_scale17', 'csbsbs_scale18', 'csbsbs_scale19', 'csbsbs_scale20'
    ]
    if any(num(row.get(k)) is None for k in required):
        return None

    g = lambda name: num(row.get(name))
    emotion = half_up(g('csbsbs_scale1') + g('csbsbs_scale2') + 3 * g('csbsbs_scale3'))
    communication = half_up(g('csbsbs_scale4') / 3 + g('csbsbs_scale5') + g('csbsbs_scale6') + g('csbsbs_scale7'))
    gestures = half_up(2 * g('csbsbs_scale8') + g('csbsbs_scale9'))
    sounds = half_up(g('csbsbs_scale10') + 2 * g('csbsbs_scale11'))
    words = half_up(g('csbsbs_scale12') + g('csbsbs_scale13') / 2 + g('csbsbs_scale14') + g('csbsbs_scale15'))
    understanding = half_up(3 * (g('csbsbs_scale16_1') + g('csbsbs_scale16_2') + g('csbsbs_scale16_3')))
    object_use = half_up(g('csbsbs_scale17') + g('csbsbs_scale18') + g('csbsbs_scale19') + g('csbsbs_scale20'))

    return {
        'csbsbs_emotionraw': emotion,
        'csbsbs_comraw': communication,
        'csbsbs_gesraw': gestures,
        'csbsbs_soundsraw': sounds,
        'csbsbs_wordsraw': words,
        'csbsbs_underraw': understanding,
        'csbsbs_objectraw': object_use,
        'csbsbs_socialcompositecalc': emotion + communication + gestures,
        'csbsbs_speechcompositecalc': sounds + words,
        'csbsbs_symboliccompositecalc': understanding + object_use,
        'csbsbs_totalrawcalc': emotion + communication + gestures + sounds + words + understanding + object_use,
    }

def cg_expected(row):
    """Calculate expected CSBS Caregiver scores."""
    # Field definitions per source notebook
    emotion_fields = [f'csbscg{i}' for i in range(1, 9)]
    communication_fields = [f'csbscg{i}' for i in range(9, 19)]
    gestures_fields = ['csbscg19'] + [f'csbscg20_{i}' for i in range(1, 11)]
    sounds_fields = ['csbscg21', 'csbscg22'] + [f'csbscg23_{i}' for i in range(1, 11)] + ['csbscg24']
    words_fields = ['csbscg25'] + [f'csbscg26_{i}' for i in range(1, 37)] + ['csbscg27', 'csbscg28']
    understanding_fields = ['csbscg29', 'csbscg30', 'csbscg31'] + [f'csbscg32_{i}' for i in range(1, 37)]
    object_fields = ['csbscg33', 'csbscg34'] + [f'csbscg35_{i}' for i in range(1, 11)] + ['csbscg36'] + [f'csbscg37_{i}' for i in range(1, 9)] + ['csbscg38', 'csbscg39'] + [f'csbscg40_{i}' for i in range(1, 11)] + [f'csbscg41_{i}' for i in range(1, 7)]
    
    # Half-weighted fields in various domains
    half_fields = {f: (lambda x: x / 2) for f in words_fields + understanding_fields + object_fields 
                   if f.split('_')[0][5:] in ('26', '32', '35', '37', '40', '41')}
    
    # Special transform for csbscg5: (2 - value)
    emotion_transforms = {'csbscg5': lambda x: 2 - x}
    
    emotion = ceil_sum(row, emotion_fields, emotion_transforms)
    communication = ceil_sum(row, communication_fields)
    gestures = ceil_sum(row, gestures_fields)
    sounds = ceil_sum(row, sounds_fields)
    words = ceil_sum(row, words_fields, half_fields)
    understanding = ceil_sum(row, understanding_fields, half_fields)
    object_use = ceil_sum(row, object_fields, half_fields)
    
    if any(v is None for v in [emotion, communication, gestures, sounds, words, understanding, object_use]):
        return None
    
    social = emotion + communication + gestures
    speech = sounds + words
    symbolic = understanding + object_use
    
    return {
        'csbs_emotionandeyegaze': emotion,
        'csbs_communication': communication,
        'csbs_gestures': gestures,
        'csbs_sounds': sounds,
        'csbs_words': words,
        'csbs_understanding': understanding,
        'csbs_objectuse': object_use,
        'csbs_socialcomposite': social,
        'csbs_speechcomposite': speech,
        'csbs_symboliccomposite': symbolic,
        'cbscg_totalscore': social + speech + symbolic,
    }

print('Utility functions and scoring logic loaded.')

Utility functions and scoring logic loaded.


## CSBS BS (PID 6207) Validation

In [3]:
print('\\n' + '='*70)
print('CSBS BS (PID 6207) - VALIDATION')
print('='*70)

metadata = post(TOKEN_6207, 'metadata')
record_id_field = metadata[0]['field_name']
score_fields = [
    'csbsbs_emotionraw','csbsbs_comraw','csbsbs_gesraw','csbsbs_soundsraw','csbsbs_wordsraw',
    'csbsbs_underraw','csbsbs_objectraw','csbsbs_socialcompositecalc','csbsbs_speechcompositecalc',
    'csbsbs_symboliccompositecalc','csbsbs_totalrawcalc'
]
input_fields = [f'csbsbs_scale{i}' for i in range(1,16)] + [
    'csbsbs_scale16_1','csbsbs_scale16_2','csbsbs_scale16_3','csbsbs_scale17','csbsbs_scale18','csbsbs_scale19','csbsbs_scale20'
]
fields = [record_id_field, 'redcap_event_name', 'csbs_bs_complete'] + input_fields + score_fields
records = post(TOKEN_6207, 'record', type='flat', forms=['csbs_bs'], fields=fields, rawOrLabel='raw', rawOrLabelHeaders='raw')

comparisons = 0
mismatches = []
for row in records:
    exp = bs_expected(row)
    if exp is None:
        continue
    rid = row.get(record_id_field, '')
    ev = row.get('redcap_event_name', '')
    for f, expected in exp.items():
        actual = num(row.get(f))
        if actual is None:
            mismatches.append({'record': rid, 'event': ev, 'field': f, 'actual': 'MISSING', 'expected': expected})
            continue
        comparisons += 1
        if abs(actual - expected) > 1e-9:
            mismatches.append({'record': rid, 'event': ev, 'field': f, 'actual': actual, 'expected': expected})

print(f'\\nRecords analyzed: {len(records)}')
print(f'Score comparisons: {comparisons}')
print(f'Mismatches found: {len(mismatches)}')

if mismatches:
    print('\\n⚠ MISMATCHES DETAIL (first 10):')
    for m in mismatches[:10]:
        print(f"  {m['record']}/{m['event']}/{m['field']}: {m['actual']} vs {m['expected']}")
    bs_validation_status = 'REVIEW REQUIRED'
else:
    print('\\n✓ PASS: All CSBS BS scores match approved logic')
    bs_validation_status = 'PASS'

\n======================================================================
CSBS BS (PID 6207) - VALIDATION


\nRecords analyzed: 1767
Score comparisons: 5533
Mismatches found: 0
\n✓ PASS: All CSBS BS scores match approved logic


## CSBS Caregiver (PID 6205) Validation

In [4]:
if TOKEN_6205:
    print('\\n' + '='*70)
    print('CSBS Caregiver (PID 6205) - VALIDATION')
    print('='*70)
    
    metadata_6205 = post(TOKEN_6205, 'metadata')
    record_id_field_6205 = metadata_6205[0]['field_name']
    cg_score_fields = [
        'csbs_emotionandeyegaze','csbs_communication','csbs_gestures','csbs_sounds',
        'csbs_words','csbs_understanding','csbs_objectuse','csbs_socialcomposite',
        'csbs_speechcomposite','csbs_symboliccomposite','cbscg_totalscore'
    ]
    
    # Get all CSBS Caregiver item fields dynamically
    all_fields_6205 = [x['field_name'] for x in metadata_6205 if x.get('form_name') == 'csbs_caregiver']
    item_fields = [f for f in all_fields_6205 if f.startswith('csbscg')]
    
    fields_6205 = [record_id_field_6205, 'redcap_event_name'] + item_fields + cg_score_fields
    records_6205 = post(TOKEN_6205, 'record', type='flat', forms=['csbs_caregiver'], fields=fields_6205, rawOrLabel='raw', rawOrLabelHeaders='raw')
    
    comparisons_6205 = 0
    mismatches_6205 = []
    for row in records_6205:
        exp = cg_expected(row)
        if exp is None:
            continue
        rid = row.get(record_id_field_6205, '')
        ev = row.get('redcap_event_name', '')
        for f, expected in exp.items():
            actual = num(row.get(f))
            if actual is None:
                mismatches_6205.append({'record': rid, 'event': ev, 'field': f, 'actual': 'MISSING', 'expected': expected})
                continue
            comparisons_6205 += 1
            if abs(actual - expected) > 1e-9:
                mismatches_6205.append({'record': rid, 'event': ev, 'field': f, 'actual': actual, 'expected': expected})
    
    print(f'\\nRecords analyzed: {len(records_6205)}')
    print(f'Score comparisons: {comparisons_6205}')
    print(f'Mismatches found: {len(mismatches_6205)}')
    
    if mismatches_6205:
        print('\\n⚠ MISMATCHES DETAIL (first 10):')
        for m in mismatches_6205[:10]:
            print(f"  {m['record']}/{m['event']}/{m['field']}: {m['actual']} vs {m['expected']}")
        cg_validation_status = 'REVIEW REQUIRED'
    else:
        print('\\n✓ PASS: All CSBS Caregiver scores match approved logic')
        cg_validation_status = 'PASS'
else:
    print('\\n⚠ CSBS Caregiver (PID 6205): SKIPPED (token not set)')
    cg_validation_status = 'SKIPPED'

\n⚠ CSBS Caregiver (PID 6205): SKIPPED (token not set)


## Field Labels & Visibility Audit

In [5]:
print('\\n' + '='*70)
print('FIELD LABELS AUDIT - AUTO-CALCULATED Prefix')
print('='*70)

print('\\n--- CSBS BS (PID 6207) ---')
metadata_6207 = post(TOKEN_6207, 'metadata')
bs_by_field = {x['field_name']: x for x in metadata_6207}
bs_score_fields = [
    'csbsbs_emotionraw','csbsbs_comraw','csbsbs_gesraw','csbsbs_soundsraw','csbsbs_wordsraw',
    'csbsbs_underraw','csbsbs_objectraw','csbsbs_socialcompositecalc','csbsbs_speechcompositecalc',
    'csbsbs_symboliccompositecalc','csbsbs_totalrawcalc'
]

bs_label_issues = []
for f in bs_score_fields:
    meta = bs_by_field.get(f)
    if not meta:
        bs_label_issues.append((f, 'FIELD NOT FOUND'))
    else:
        label = meta.get('field_label', '')
        ftype = meta.get('field_type', '')
        has_auto = label.startswith('AUTO-CALCULATED:')
        if has_auto and ftype == 'calc':
            print(f'  ✓ {f:<30} : {label[:60]}')
        else:
            issue = f'Missing AUTO-CALCULATED prefix' if not has_auto else f'Type={ftype} (expected calc)'
            bs_label_issues.append((f, issue))

if bs_label_issues:
    print('\\n  ⚠ Issues found:')
    for f, issue in bs_label_issues:
        print(f'    {f}: {issue}')
else:
    print('\\n  ✓ All CSBS BS score fields properly labeled with AUTO-CALCULATED prefix')

if TOKEN_6205:
    print('\\n--- CSBS Caregiver (PID 6205) ---')
    cg_by_field = {x['field_name']: x for x in metadata_6205}
    cg_score_fields = [
        'csbs_emotionandeyegaze','csbs_communication','csbs_gestures','csbs_sounds',
        'csbs_words','csbs_understanding','csbs_objectuse','csbs_socialcomposite',
        'csbs_speechcomposite','csbs_symboliccomposite','cbscg_totalscore'
    ]
    
    cg_label_issues = []
    for f in cg_score_fields:
        meta = cg_by_field.get(f)
        if not meta:
            cg_label_issues.append((f, 'FIELD NOT FOUND'))
        else:
            label = meta.get('field_label', '')
            ftype = meta.get('field_type', '')
            has_auto = label.startswith('AUTO-CALCULATED:')
            if has_auto and ftype == 'calc':
                print(f'  ✓ {f:<30} : {label[:60]}')
            else:
                issue = f'Missing AUTO-CALCULATED prefix' if not has_auto else f'Type={ftype} (expected calc)'
                cg_label_issues.append((f, issue))
    
    if cg_label_issues:
        print('\\n  ⚠ Issues found:')
        for f, issue in cg_label_issues:
            print(f'    {f}: {issue}')
    else:
        print('\\n  ✓ All CSBS Caregiver score fields properly labeled with AUTO-CALCULATED prefix')

\n======================================================================
FIELD LABELS AUDIT - AUTO-CALCULATED Prefix
\n--- CSBS BS (PID 6207) ---


  ✓ csbsbs_emotionraw              : AUTO-CALCULATED: Emotion and Eye Gaze Weighted Raw Score
  ✓ csbsbs_comraw                  : AUTO-CALCULATED: Communication Weighted Raw Score
  ✓ csbsbs_gesraw                  : AUTO-CALCULATED: Gestures Weighted Raw Score
  ✓ csbsbs_soundsraw               : AUTO-CALCULATED: Sounds Weighted Raw Score
  ✓ csbsbs_wordsraw                : AUTO-CALCULATED: Words Weighted Raw Score
  ✓ csbsbs_underraw                : AUTO-CALCULATED: Understanding Weighted Raw Score
  ✓ csbsbs_objectraw               : AUTO-CALCULATED: Object Use Weighted Raw Score
  ✓ csbsbs_socialcompositecalc     : AUTO-CALCULATED: Social Composite Score
  ✓ csbsbs_speechcompositecalc     : AUTO-CALCULATED: Speech Composite Score
  ✓ csbsbs_symboliccompositecalc   : AUTO-CALCULATED: Symbolic Composite Score
  ✓ csbsbs_totalrawcalc            : AUTO-CALCULATED: CSBS BS Total Raw Score
\n  ✓ All CSBS BS score fields properly labeled with AUTO-CALCULATED prefix


## Consistency Check & Summary

In [6]:
print('\\n' + '='*70)
print('COMPREHENSIVE VALIDATION SUMMARY')
print('='*70)

print('\\n1. SCORING LOGIC VALIDATION:')
print(f'   CSBS BS (PID 6207):           {bs_validation_status}')
if TOKEN_6205:
    print(f'   CSBS Caregiver (PID 6205):    {cg_validation_status}')
else:
    print(f'   CSBS Caregiver (PID 6205):    ⚠ {cg_validation_status}')

print('\\n2. FIELD LABELING:')
if not bs_label_issues:
    print(f'   CSBS BS (PID 6207):           ✓ All fields properly labeled')
else:
    print(f'   CSBS BS (PID 6207):           ⚠ {len(bs_label_issues)} issues')

if TOKEN_6205:
    if not cg_label_issues:
        print(f'   CSBS Caregiver (PID 6205):    ✓ All fields properly labeled')
    else:
        print(f'   CSBS Caregiver (PID 6205):    ⚠ {len(cg_label_issues)} issues')

print('\\n3. RECORD COVERAGE:')
print(f'   CSBS BS: {len(records)} records analyzed')
if TOKEN_6205:
    print(f'   CSBS Caregiver: {len(records_6205)} records analyzed')

print('\\n' + '='*70)
if TOKEN_6205:
    overall = 'PASS' if (bs_validation_status == 'PASS' and cg_validation_status == 'PASS' and not bs_label_issues and not cg_label_issues) else 'REVIEW REQUIRED'
else:
    overall = 'PARTIAL' if bs_validation_status == 'PASS' and not bs_label_issues else 'REVIEW REQUIRED'

print(f'OVERALL VALIDATION STATUS: {overall}')
if not TOKEN_6205:
    print('\\nTo complete full dual-project validation, set:')
    print('  export REDCAP_TOKEN_6205=<your_token>')
print('='*70)

\n======================================================================
COMPREHENSIVE VALIDATION SUMMARY
\n1. SCORING LOGIC VALIDATION:
   CSBS BS (PID 6207):           PASS
   CSBS Caregiver (PID 6205):    ⚠ SKIPPED
\n2. FIELD LABELING:
   CSBS BS (PID 6207):           ✓ All fields properly labeled
\n3. RECORD COVERAGE:
   CSBS BS: 1767 records analyzed
\n======================================================================
OVERALL VALIDATION STATUS: PARTIAL
\nTo complete full dual-project validation, set:
  export REDCAP_TOKEN_6205=<your_token>
